<a href="https://colab.research.google.com/github/maaviarizwan/skin-lesion-research/blob/main/notebooks/Stage3_Baseline_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synaptix Research — Skin Lesion Classification
## Stage 3: Baseline Models (EfficientNet-B3 vs ViT-Small)



**What this stage does:**
- Trains EfficientNet-B3 (CNN baseline)
- Trains ViT-Small (Transformer baseline)
- Evaluates both with Accuracy, F1, AUC-ROC, Confusion Matrix
- Saves all results to Drive for paper

**Expected runtime:** ~2.5 hours total on T4 GPU
---

## Cell 1 — Install Libraries + Mount Drive

In [ ]:
!pip install timm albumentations kaggle kagglehub scikit-learn tqdm -q

from google.colab import drive
drive.mount('/content/drive')

RESEARCH_DIR = '/content/drive/MyDrive/Synaptix_Research'
import os
os.makedirs(f'{RESEARCH_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{RESEARCH_DIR}/results',     exist_ok=True)
os.makedirs(f'{RESEARCH_DIR}/figures',     exist_ok=True)

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Drive mounted')

## Cell 2 — All Imports

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS = 25
PATIENCE   = 7    # early stopping

LESION_NAMES = {
    'nv':'Melanocytic Nevi', 'mel':'Melanoma',
    'bkl':'Benign Keratosis', 'bcc':'Basal Cell Carcinoma',
    'akiec':'Actinic Keratoses', 'vasc':'Vascular Lesions', 'df':'Dermatofibroma'
}
print(f'Device: {device}')
print('All imports done')

## Cell 3 — Download Dataset (Session-Safe)

Colab wipes /content/ every session. This cell re-downloads images. The CSV and splits are already saved on Drive from Stage 2.

In [ ]:
import kagglehub

# ── Paste your Kaggle token here ──
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_c7798b0fe3d02db26a83507e2c7e334e'

print('Downloading HAM10000 (3-5 minutes)...')
path = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')
print(f'Downloaded to: {path}')

# Find all images recursively
image_path_dict = {}
for root, dirs, files in os.walk(path):
    for filename in files:
        if filename.lower().endswith(('.jpg','.jpeg','.png')):
            img_id = os.path.splitext(filename)[0]
            image_path_dict[img_id] = os.path.join(root, filename)

print(f'Images found: {len(image_path_dict)}')
if len(image_path_dict) == 0:
    raise RuntimeError('No images found. Check your token.')
print('Dataset ready')

## Cell 4 — Load Splits + Config from Drive

Loads the exact train/val/test splits saved in Stage 2. This guarantees we train and evaluate on the same data.

In [ ]:
# Load config saved in Stage 2
with open(f'{RESEARCH_DIR}/config.json', 'r') as f:
    config = json.load(f)

label_map        = config['label_map']
class_weights    = np.array(config['class_weights'])
NUM_CLASSES      = config['num_classes']
IMAGENET_MEAN    = config['imagenet_mean']
IMAGENET_STD     = config['imagenet_std']
idx_to_cls       = {v: k for k, v in label_map.items()}

# Load splits
train_df = pd.read_csv(f'{RESEARCH_DIR}/split_train.csv')
val_df   = pd.read_csv(f'{RESEARCH_DIR}/split_val.csv')
test_df  = pd.read_csv(f'{RESEARCH_DIR}/split_test.csv')

# Remap image paths to this session
for frame in [train_df, val_df, test_df]:
    frame['path'] = frame['image_id'].map(image_path_dict)

train_df = train_df[train_df['path'].notna()].reset_index(drop=True)
val_df   = val_df[val_df['path'].notna()].reset_index(drop=True)
test_df  = test_df[test_df['path'].notna()].reset_index(drop=True)

class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Classes: {NUM_CLASSES}')
print('Config and splits loaded')

## Cell 5 — Dataset Class + DataLoaders

In [ ]:
# Transforms
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.HueSaturationValue(10, 20, 10, p=0.4),
    A.GridDistortion(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()
])
val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()
])

class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = np.array(Image.open(row['path']).convert('RGB'))
        label = int(row['label'])
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label

# Sampler
sample_weights = torch.DoubleTensor(
    [class_weights[l] for l in train_df['label']]
)
sampler = WeightedRandomSampler(sample_weights, len(train_df), replacement=True)

# DataLoaders
train_loader = DataLoader(HAM10000Dataset(train_df, train_transform),
                          batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(HAM10000Dataset(val_df,   val_test_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(HAM10000Dataset(test_df,  val_test_transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')
print('DataLoaders ready')

## Cell 6 — Training Utilities

Reusable functions used for both models. Don't change anything here.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct    += model(images).argmax(1).eq(labels).sum().item()
        total      += labels.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            probs   = torch.softmax(outputs, dim=1)
            loss    = criterion(outputs, labels)
            preds   = outputs.argmax(1)
            total_loss += loss.item() * images.size(0)
            correct    += preds.eq(labels).sum().item()
            total      += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    return (total_loss/total, correct/total,
            np.array(all_preds), np.array(all_labels), np.array(all_probs))

def compute_metrics(preds, labels, probs, split='val'):
    acc    = accuracy_score(labels, preds)
    f1     = f1_score(labels, preds, average='weighted')
    try:
        auc = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except:
        auc = 0.0
    print(f'  [{split}] Acc={acc:.4f}  F1={f1:.4f}  AUC={auc:.4f}')
    return {'accuracy': acc, 'f1_weighted': f1, 'auc_roc': auc}

def train_model(model, model_name, train_loader, val_loader,
                optimizer, scheduler, criterion, device,
                epochs=NUM_EPOCHS, patience=PATIENCE):
    best_val_f1   = 0.0
    patience_ctr  = 0
    history       = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[], 'val_f1':[]}
    ckpt_path     = f'{RESEARCH_DIR}/checkpoints/{model_name}_best.pth'

    print(f'\nTraining {model_name} for up to {epochs} epochs...')
    print(f'Checkpoint saves to: {ckpt_path}')
    print('-' * 60)

    for epoch in range(1, epochs+1):
        t0 = time.time()

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, vp, vl, vprob = evaluate(model, val_loader, criterion, device)
        val_f1 = f1_score(vl, vp, average='weighted')

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        elapsed = time.time() - t0
        print(f'Epoch {epoch:02d}/{epochs} | '
              f'Loss {train_loss:.4f}/{val_loss:.4f} | '
              f'Acc {train_acc:.4f}/{val_acc:.4f} | '
              f'F1 {val_f1:.4f} | {elapsed:.0f}s')

        # Save best checkpoint
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), ckpt_path)
            print(f'  --> New best F1: {best_val_f1:.4f} — checkpoint saved')
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f'\nEarly stopping at epoch {epoch} (no improvement for {patience} epochs)')
                break

    print(f'\nTraining complete. Best val F1: {best_val_f1:.4f}')
    return history, ckpt_path

print('Training utilities ready')

## Cell 7 — Plot Utility

In [ ]:
def plot_training_curves(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=13, fontweight='bold')

    epochs = range(1, len(history['train_loss'])+1)

    axes[0].plot(epochs, history['train_loss'], label='Train', color='royalblue')
    axes[0].plot(epochs, history['val_loss'],   label='Val',   color='tomato')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    axes[1].plot(epochs, history['train_acc'], label='Train', color='royalblue')
    axes[1].plot(epochs, history['val_acc'],   label='Val',   color='tomato')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    axes[2].plot(epochs, history['val_f1'], color='green')
    axes[2].set_title('Val F1-Score'); axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    save_path = f'{RESEARCH_DIR}/figures/{model_name}_curves.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

def plot_confusion_matrix(labels, preds, model_name):
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=list(label_map.keys()),
                yticklabels=list(label_map.keys()))
    plt.title(f'{model_name} — Confusion Matrix (Test Set)', fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout()
    save_path = f'{RESEARCH_DIR}/figures/{model_name}_confusion.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

print('Plot utilities ready')

## Cell 8 — Model 1: EfficientNet-B3 (CNN Baseline)

EfficientNet-B3 is our CNN baseline. Pretrained on ImageNet, we replace the final layer with a 7-class head and fine-tune the whole network.

**Expected time:** ~60-75 minutes on T4

In [ ]:
# Build EfficientNet-B3
eff_model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=NUM_CLASSES)
eff_model = eff_model.to(device)

total_params = sum(p.numel() for p in eff_model.parameters()) / 1e6
print(f'EfficientNet-B3 parameters: {total_params:.1f}M')

# Optimizer + Scheduler + Loss
eff_optimizer  = torch.optim.AdamW(eff_model.parameters(), lr=1e-4, weight_decay=1e-2)
eff_scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(eff_optimizer, T_max=NUM_EPOCHS)
criterion      = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Train
eff_history, eff_ckpt = train_model(
    eff_model, 'EfficientNet_B3',
    train_loader, val_loader,
    eff_optimizer, eff_scheduler, criterion, device
)

# Plot training curves
plot_training_curves(eff_history, 'EfficientNet_B3')

## Cell 9 — Evaluate EfficientNet-B3 on Test Set

We load the best checkpoint (saved during training) and evaluate on the held-out test set.

In [ ]:
# Load best checkpoint
eff_model.load_state_dict(torch.load(
    f'{RESEARCH_DIR}/checkpoints/EfficientNet_B3_best.pth',
    map_location=device
))

# Evaluate on test set
_, _, eff_preds, eff_labels, eff_probs = evaluate(
    eff_model, test_loader, criterion, device
)

print('EfficientNet-B3 — Test Set Results:')
print('=' * 45)
eff_metrics = compute_metrics(eff_preds, eff_labels, eff_probs, split='test')

print('\nPer-class Report:')
print(classification_report(
    eff_labels, eff_preds,
    target_names=list(label_map.keys())
))

plot_confusion_matrix(eff_labels, eff_preds, 'EfficientNet_B3')

## Cell 10 — Model 2: ViT-Small (Transformer Baseline)

ViT-Small is our Vision Transformer baseline. Same setup as EfficientNet — pretrained ImageNet weights, 7-class head, full fine-tuning.

**Expected time:** ~75-90 minutes on T4

In [ ]:
# Build ViT-Small
vit_model = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=NUM_CLASSES)
vit_model = vit_model.to(device)

total_params = sum(p.numel() for p in vit_model.parameters()) / 1e6
print(f'ViT-Small parameters: {total_params:.1f}M')

# Optimizer + Scheduler + Loss
vit_optimizer = torch.optim.AdamW(vit_model.parameters(), lr=1e-4, weight_decay=1e-2)
vit_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(vit_optimizer, T_max=NUM_EPOCHS)

# Train
vit_history, vit_ckpt = train_model(
    vit_model, 'ViT_Small',
    train_loader, val_loader,
    vit_optimizer, vit_scheduler, criterion, device
)

# Plot training curves
plot_training_curves(vit_history, 'ViT_Small')

## Cell 11 — Evaluate ViT-Small on Test Set

In [ ]:
# Load best checkpoint
vit_model.load_state_dict(torch.load(
    f'{RESEARCH_DIR}/checkpoints/ViT_Small_best.pth',
    map_location=device
))

# Evaluate
_, _, vit_preds, vit_labels, vit_probs = evaluate(
    vit_model, test_loader, criterion, device
)

print('ViT-Small — Test Set Results:')
print('=' * 45)
vit_metrics = compute_metrics(vit_preds, vit_labels, vit_probs, split='test')

print('\nPer-class Report:')
print(classification_report(
    vit_labels, vit_preds,
    target_names=list(label_map.keys())
))

plot_confusion_matrix(vit_labels, vit_preds, 'ViT_Small')

## Cell 12 — Side-by-Side Comparison

This table goes directly into your paper as Table 1.

In [ ]:
print('=' * 55)
print('  BASELINE RESULTS — TABLE 1 (for paper)')
print('=' * 55)
print(f'{"Model":<20} {"Accuracy":>10} {"F1 (W)":>10} {"AUC-ROC":>10}')
print('-' * 55)
print(f'{"EfficientNet-B3":<20} '
      f'{eff_metrics["accuracy"]:>10.4f} '
      f'{eff_metrics["f1_weighted"]:>10.4f} '
      f'{eff_metrics["auc_roc"]:>10.4f}')
print(f'{"ViT-Small":<20} '
      f'{vit_metrics["accuracy"]:>10.4f} '
      f'{vit_metrics["f1_weighted"]:>10.4f} '
      f'{vit_metrics["auc_roc"]:>10.4f}')
print('=' * 55)
print('Our Hybrid model (Stage 4) should outperform both')

# Bar chart comparison
metrics_names = ['Accuracy', 'F1 (Weighted)', 'AUC-ROC']
eff_vals = [eff_metrics['accuracy'], eff_metrics['f1_weighted'], eff_metrics['auc_roc']]
vit_vals = [vit_metrics['accuracy'], vit_metrics['f1_weighted'], vit_metrics['auc_roc']]

x = np.arange(len(metrics_names))
fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - 0.2, eff_vals, 0.35, label='EfficientNet-B3', color='royalblue')
bars2 = ax.bar(x + 0.2, vit_vals, 0.35, label='ViT-Small',       color='tomato')
ax.set_xticks(x); ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score'); ax.legend(fontsize=11)
ax.set_title('Baseline Model Comparison — Test Set', fontsize=13, fontweight='bold')
for bar in list(bars1)+list(bars2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
save_path = f'{RESEARCH_DIR}/figures/baseline_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

## Cell 13 — Save All Results to Drive

In [ ]:
results = {
    'EfficientNet_B3': {**eff_metrics, 'history': eff_history},
    'ViT_Small':       {**vit_metrics, 'history': vit_history}
}

with open(f'{RESEARCH_DIR}/results/stage3_baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Results saved to Drive')
print(f'  {RESEARCH_DIR}/results/stage3_baseline_results.json')

## Cell 14 — Stage 3 Complete

In [ ]:
print('=' * 55)
print('  STAGE 3 COMPLETE')
print('=' * 55)
print(f'  EfficientNet-B3 Acc : {eff_metrics["accuracy"]:.4f}')
print(f'  EfficientNet-B3 F1  : {eff_metrics["f1_weighted"]:.4f}')
print(f'  ViT-Small Acc       : {vit_metrics["accuracy"]:.4f}')
print(f'  ViT-Small F1        : {vit_metrics["f1_weighted"]:.4f}')
print()
print('  Saved to Drive:')
print('    checkpoints/EfficientNet_B3_best.pth')
print('    checkpoints/ViT_Small_best.pth')
print('    results/stage3_baseline_results.json')
print('    figures/baseline_comparison.png')
print()
print('  NEXT: Stage 4 — Hybrid CNN-Transformer Model')
print('=' * 55)